# RF Board — PYNQ Main Notebook

Direct hardware control on the Zynq-7020 ARM PS (no TCP).

**Sequence** (mirrors `PC_code/main.py`):
1. Load PL bitstream overlay
2. Instantiate `PynqController`
3. Read ADC chip identification
4. Configure CDCE62005 clock distributor
5. Initialize DACs with default settings
6. **BRAM profile control** — init, write frequency profiles, write amplitude ramp profiles
7. (Optional) Configure NCO frequencies via PL DDS

In [ ]:
# Step 1 — Load the PL bitstream overlay
from pynq import Overlay
ol = Overlay('./projectHW_MT.xsa')
print('Overlay loaded.')

In [ ]:
# Step 2 — Instantiate controller (set BOARD_VER = 1 or 2)
# ol is passed so BRAMController can resolve MT_0 / MT_1 memory aliases.
from pynq_controller import PynqController

BOARD_VER = 2
zynq = PynqController(ol, board_ver=BOARD_VER)

In [ ]:
# Step 3 — Read ADC chip identification registers
print('\n[ADC] Reading chip identification registers:')
spi_port_config = zynq.adc_read_register(0x00)  # SPI port config
chip_id         = zynq.adc_read_register(0x01)  # Chip ID (expected: 0x82)
chip_grade      = zynq.adc_read_register(0x02)  # Chip grade
print(f'  SPI config : 0x{spi_port_config:02X}')
print(f'  Chip ID    : 0x{chip_id:02X}  (expected 0x82)')
print(f'  Chip grade : 0x{chip_grade:02X}')

In [ ]:
# Step 4 — Configure CDCE62005 clock distributor
# DAC 1 GHz, ADC 250 MHz (use with DACCLK_HZ = 1_000_000_000 below)
CDCE_CONFIG = '8340032083400301EB060302680603036806031410000BE504BE09F6BD0037F7'

# Alternative configs (uncomment to use):
# DAC 100 MHz, ADC 100 MHz:
# CDCE_CONFIG = '830603208306030183060302EB060003EB06001410000BE504BE09E6BD0037F7'

zynq.configure_clock(CDCE_CONFIG)

# Update DACCLK_HZ to match the DAC clock frequency in the config above
zynq.DACCLK_HZ = 1_000_000_000   # 1 GHz matches the config above
# zynq.DACCLK_HZ = 100_000_000   # uncomment for 100 MHz config

In [ ]:
# Step 5 — Initialize DACs with default settings
zynq.dac_init()

---
## Step 6 — BRAM profile control

Two IQ channels for DAC1, each with its own frequency and amplitude BRAM:

| Channel | DAC1 pair | Overlay alias (freq)                        | Overlay alias (amp)   |
|---------|-----------|---------------------------------------------|-----------------------|
| 1       | A/B       | `MT_0IQ32tones_w_bramaxi_bram_ctrl_0`       | `MT_0bram_AMP_ctrl`   |
| 2       | C/D       | `MT_1IQ32tones_w_bramaxi_bram_ctrl_0`       | `MT_1bram_AMP_ctrl`   |

**Frequency BRAM** — up to 8 profiles × 32 tones each.  
Each tone: `(amplitude 0–1, freq_mhz, phase_deg)`.

**Amplitude BRAM** — on/off ramp sequences.  
Each ramp segment: `(num_steps, step_size)` where `step_size` is signed 16-bit.

`profile_base_addr` and `AM_onoff` are driven externally by another board via FPGA pins — Python only loads the BRAM data.

In [ ]:
# Step 6a — Initialize BRAMs (clears amplitude pointer tables, resets allocators)
# Run once after overlay load, before writing any profiles.
zynq.bram_init()

In [ ]:
# Step 6b — Write frequency profiles to BRAM
#
# Each profile holds up to 32 tones: (amplitude 0-1, freq_mhz, phase_deg)
# profile_num : 0-7
# channel     : 1 = DAC1 AB (MT_0),  2 = DAC1 CD (MT_1)
#
# The external board drives profile_base_addr to select the active profile.

# --- Define tone sets (edit freely) ---
profile0 = [(1.0,  5.0,   0)]              # single 5 MHz tone, full amplitude
profile1 = [(1.0,  5.0,   0),              # two tones: 5 MHz + 10 MHz
            (1.0, 10.0,   0)]
profile2 = [(1.0,  5.0,   0),              # same freq, 90° phase shift
            (1.0,  5.0,  90)]
profile3 = [(1.0,  1.0,   0),              # four-tone comb
            (1.0,  5.0,   0),
            (1.0, 10.0,   0),
            (1.0, 20.0,   0)]

# --- Write to both channels ---
for ch in (1, 2):
    zynq.set_freq_profile(channel=ch, profile_num=0, data=profile0, DEBUG=0)
    zynq.set_freq_profile(channel=ch, profile_num=1, data=profile1, DEBUG=0)
    zynq.set_freq_profile(channel=ch, profile_num=2, data=profile2, DEBUG=0)
    zynq.set_freq_profile(channel=ch, profile_num=3, data=profile3, DEBUG=0)
    print(f"✓ CH{ch}: 4 frequency profiles written")

print("\nDone.")

In [ ]:
# Step 6c — Write amplitude ramp profiles to BRAM
#
# Each profile has an ON ramp (amplitude rises) and an OFF ramp (amplitude falls).
# Ramp entry: (num_steps, step_size)
#   num_steps : 0–65535 — clock ticks per step
#   step_size : −32768 … +32767 — amplitude delta per tick
#               positive → ramp up (ON),  negative → ramp down (OFF)
#
# The external board drives AM_onoff to trigger the ramp transition.
# Amplitude register is 16-bit: 0 = silent, 32767 = full scale.

NUM_STEPS = 500   # ramp duration in clock ticks

# --- Define ramp shape (one or more segments) ---
on_ramp  = [(NUM_STEPS,  1000)]   # linear rise:  500 steps × +1000
off_ramp = [(NUM_STEPS, -1000)]   # linear fall:  500 steps × -1000

# Multi-segment example (uncomment to use):
# on_ramp  = [(200, 500), (300, 2000)]   # slow start, faster finish
# off_ramp = [(500, -1000)]

# --- Write to both channels (independent allocators) ---
for ch in (1, 2):
    alloc = zynq.get_amp_allocator(channel=ch)
    for profile_num in range(4):     # profiles 0-3, matching freq profiles above
        zynq.set_amp_profile(
            channel=ch,
            profile_num=profile_num,
            on_ramps=on_ramp,
            off_ramps=off_ramp,
            allocator=alloc,
            DEBUG=0,
        )
    print(f"✓ CH{ch}: 4 amplitude profiles written")

print("\nDone.")

In [ ]:
# Step 6 — Configure DAC2 DDS (PL fabric)
from pynq_dds import DAC2_DDS

dds2 = DAC2_DDS(ol)

# Set frequency and phase for both channel pairs independently
dds2.set(ab_freq_mhz=5.0,  ab_phase_deg=0.0,
         cd_freq_mhz=5.0,  cd_phase_deg=0.0)

# Different frequencies:
# dds2.set(ab_freq_mhz=5.0,  ab_phase_deg=0.0,
#          cd_freq_mhz=10.0, cd_phase_deg=0.0)

# Same frequency, 90° phase shift between A/B and C/D:
# dds2.set(ab_freq_mhz=5.0, ab_phase_deg=0.0,
#          cd_freq_mhz=5.0,  cd_phase_deg=90.0)

# Stop output:
# dds2.off()

---
## Diagnostic / debug cells

In [ ]:
# Read back all CDCE registers
regs = zynq.cdce.read_all()

In [ ]:
# Read back all DAC registers
zynq.read_all_dac_registers(dac_num=1)
zynq.read_all_dac_registers(dac_num=2)

In [ ]:
# Read back key ADC registers
zynq.adc_read_all_registers()

In [ ]:
# DAC SYNC (disable → quad mode → re-enable both DACs)
zynq.DAC_SYNC()

### BRAM diagnostics

In [ ]:
# Read back a frequency profile from BRAM and print it
CH      = 1     # 1 = DAC1 AB (MT_0),  2 = DAC1 CD (MT_1)
PROFILE = 0     # 0-7

profile_data = zynq.get_freq_profile(channel=CH, profile_num=PROFILE)
if profile_data:
    print(f"\nFrequency profile CH{CH} / profile {PROFILE}:")
    for i, (amp, freq, phase) in enumerate(profile_data):
        print(f"  [{i}]  amp={amp:.3f}  freq={freq:.3f} MHz  phase={phase:.1f}°")

In [ ]:
# Dump the amplitude BRAM pointer table for one channel
# Shows ON/OFF ramp address for each profile
zynq.bram.dump_pointer_table(channel=1)   # 1=MT_0/AB,  2=MT_1/CD

In [ ]:
# Print full memory allocation map for a channel's amplitude allocator
zynq.get_amp_allocator(channel=1).print_allocation_map()   # 1=MT_0/AB,  2=MT_1/CD

In [ ]:
# Raw BRAM word dump — cross-check against ILA / VHDL simulation
CH     = 1       # 1=MT_0/AB,  2=MT_1/CD
OFFSET = 0x000   # 0x000 = start of profile 0;  0x100 = start of profile 1

raw = zynq.bram.read_freq_bram_raw(channel=CH, n_words=8, byte_offset=OFFSET)
print(f"Freq BRAM CH{CH} ({zynq.bram.FREQ_MEM_NAMES[CH]}) @ 0x{OFFSET:04X}:")
for i, w in enumerate(raw):
    print(f"  [0x{OFFSET + i*4:04X}]  0x{w:08X}")